In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
!apt-get update
!apt-get install poppler-utils
!apt-get install tesseract-ocr
!apt-get install tesseract-ocr-ben  # Support Bengali also

# openpyxl for save as exxcel file
!pip install pdf2image pytesseract pandas openpyxl

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly install

In [64]:
import os
import glob
from pdf2image import convert_from_path
import pytesseract
import pandas as pd

In [65]:
def extract_text_from_pdf(file_path):
    # Convert PDF pages into 300 DPI images
    pages = convert_from_path(file_path, 300)

    full_text = ""
    # Extracting tect from each pages
    for page in pages:
        # lang='ben+eng' read both languages English and Bengali
        text = pytesseract.image_to_string(page, lang='ben+eng')
        full_text += text + "\n"

    return full_text.strip()

In [66]:
folder_path = ("/content/drive/MyDrive/Result/")

In [67]:
# Listing all PDF file inside the folder
pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))

In [68]:
print(f"Total {len(pdf_files)} PDF files got. Precess starting...\n")

Total 6 PDF files got. Precess starting...



In [70]:
# Create an empty list to storing data
extracted_data = []

# Loop for each PDF diles
for pdf_path in pdf_files:
    file_name = os.path.basename(pdf_path) # file name
    print(f"Precessing: {file_name} ...")
    try:
        # Extract text
        extracted_text = extract_text_from_pdf(pdf_path)
        # Add file name and extraced text into list
        extracted_data.append({
            "File Name": file_name,
            "Extracted Text": extracted_text
        })
        print(f"Sucessfully text extracted from {file_name}\n")
    except Exception as e:
        print(f"Error! {file_name} process not complete because: {e}\n")


Precessing: UUOnlineStudentGradeMarksheet (5).pdf ...
Sucessfully text extracted from UUOnlineStudentGradeMarksheet (5).pdf

Precessing: UUOnlineStudentGradeMarksheet (4).pdf ...
Sucessfully text extracted from UUOnlineStudentGradeMarksheet (4).pdf

Precessing: UUOnlineStudentGradeMarksheet (3).pdf ...
Sucessfully text extracted from UUOnlineStudentGradeMarksheet (3).pdf

Precessing: UUOnlineStudentGradeMarksheet (2).pdf ...
Sucessfully text extracted from UUOnlineStudentGradeMarksheet (2).pdf

Precessing: UUOnlineStudentGradeMarksheet (1).pdf ...
Sucessfully text extracted from UUOnlineStudentGradeMarksheet (1).pdf

Precessing: UUOnlineStudentGradeMarksheet (6).pdf ...
Sucessfully text extracted from UUOnlineStudentGradeMarksheet (6).pdf



In [71]:
# Creating Pandas DataFrame by listed data
df = pd.DataFrame(extracted_data)

In [39]:
# Where the excel file will be saved
excel_output_path = ("/content/drive/MyDrive/Excel files/Extracted_Data.xlsx")

In [40]:
# DataFrame saving as a excel file
df.to_excel(excel_output_path, index=False, engine='openpyxl')

In [72]:
print(f" Done. Data sucessfully saved in excel")
print(f"Excel file is here: {excel_output_path}")

 Done. Data sucessfully saved in excel
Excel file is here: /content/drive/MyDrive/Excel files/Extracted_Data.xlsx


In [43]:
import re

In [45]:
# 1 read excel
file_path = ("/content/drive/MyDrive/Excel files/Extracted_Data.xlsx")
df = pd.read_excel(file_path)

In [47]:
all_subjects = []
# Creating a map acording to Semister (I, II, III...)
sem_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6}

In [51]:
# 2. Scan each (Row) or  marksheets text
for index, row in df.iterrows():
    text = str(row['Extracted Text'])
    # Extraxting semister's name (such: I, II, III)
    sem_match = re.search(r'- ([IVX]+) SEMESTER', text)
    semester = sem_match.group(1) if sem_match else "Unknown"
    # Split all text according to lines
    lines = text.split('\n')

    for line in lines:
        line = line.strip()

        # 3. Searching subject and marks by Regex pattern
        # (It will search lines which contain number in its end, such: 85/100)
        match = re.search(r'^(.*?)\s+(?:(?:\d+\s+){0,2})?(\d+/\d+)', line)

        if match:
            subject = match.group(1).strip()
            marks = match.group(2).strip()

            # Erase extra 'OBCA' or 'OBCA-' part before subject's name
            subject = re.sub(r'^OBCA[-\s]*', '', subject, flags=re.IGNORECASE).strip()

            # Saving data in list
            all_subjects.append({
                'Semester': semester,
                'Subject Name': subject,
                'Marks Obtained': marks,
                'Sem_Num': sem_map.get(semester, 99) # for sort
            })

In [52]:
# 4. List to dataframe table
subjects_df = pd.DataFrame(all_subjects)

In [53]:
# Sorting data according to Semister and delete 'Sem_Num' column
subjects_df = subjects_df.sort_values(by=['Sem_Num']).drop(columns=['Sem_Num'])

In [54]:
# Fix table index and sereal no
subjects_df = subjects_df.reset_index(drop=True)

In [59]:
# 5. Saving final data as CSV
output_path =("/content/drive/MyDrive/Datasets CSV/Subject_Wise_Marks.csv")
subjects_df.to_csv(output_path, index=False)

In [73]:
print("Data sucessfully saved as a structured way:", output_path)

Data sucessfully saved as a structured way: /content/drive/MyDrive/Datasets CSV/Subject_Wise_Marks.csv


In [61]:
data = pd.read_csv("/content/drive/MyDrive/Datasets CSV/Subject_Wise_Marks.csv")

In [63]:
data.head(30)

,Semester,Subject Name,Marks Obtained
0,I,INTRODUCTION TO INFORMATIVE,75/100
1,I,DIGITAL COMPUTER AND,66/100
2,I,MATHEMATICS,62/100
3,I,PROGRAMMING IN C,85/100
4,I,COMMUNICATIVE ENGLISH,79/100
5,II,DISCRETE MATHEMATICS,80/100
6,II,ENVIRONMENTAL STUDIES,82/100
7,II,DATA STRUCTURE USING C,90/100
8,II,CORE JAVA,91/100
9,II,ACCOUNTING AND FINANCIAL,84/100
